# Language-Model Adapter — Artifact Inference Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_artifact_inference_colab.ipynb)

This companion tutorial demonstrates how to load, verify, and run inference using an exported PEFT adapter package.

> upload adapter ZIP → verify manifest/hashes/provenance → resolve exact base model → attach adapter → generate

---

## The Tripartite Architecture at Inference Time

In a parameter-efficient fine-tuning workflow, the trained adapter is completely decoupled from the base model:
- **Base Model ($W_0$):** Retains full pre-trained linguistic knowledge and reasoning capacity across billions of frozen parameters.
- **Adapter Matrices ($\Delta W = \frac{\alpha}{r}BA$):** Lightweight delta weights (typically 50–150 MB) capturing task-specific instruction adherence, tone, and formatting.
- **Tokenizer & Chat Template:** Preserved within the adapter package to ensure prompt formatting matches the fine-tuning token distribution exactly.

The original training dataset is not needed during inference. All you need is the lightweight adapter package and access to the exact base model checkpoint it was trained against.


## 1. Runtime and Artifact Verification

To run inference, we install the core generation stack (`transformers`, `peft`, `accelerate`, and `bitsandbytes` for 4-bit loading).

When loading an external model artifact, it is best practice to verify its integrity before unpacking. The code below performs strict safety checks on the uploaded adapter ZIP:
- **Path Sanitization**: Rejects path traversal (`..`), absolute paths, and backslashes to prevent directory escape;
- **Symlink Protection**: Refuses any symbolic links to avoid link-based attacks; and
- **Cryptographic Hashing**: Reads `artifact-manifest.json` and checks that every file on disk matches its recorded size and SHA-256 digest.


In [ ]:
%pip -q install transformers==4.57.1 tokenizers==0.22.1 huggingface-hub==0.36.0 peft==0.18.0 accelerate==1.11.0 bitsandbytes==0.49.0 safetensors==0.8.0


In [ ]:
import hashlib,json,re,shutil,stat,zipfile
from pathlib import Path,PurePosixPath
import pandas as pd,torch
from huggingface_hub import HfApi
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import PeftModel
AI_PROVENANCE={"provider":"OpenAI","product":"ChatGPT","model":"GPT-5.6 Sol High","role":"Builder"}
if not torch.cuda.is_available(): raise RuntimeError("Use a Colab GPU runtime")
EXPECTED_ARTIFACT_ZIP_SHA256 = "" # @param {type:"string"}
from google.colab import files
u=files.upload()
if len(u)!=1: raise ValueError("Upload exactly one adapter ZIP")
zp=Path("/content")/next(iter(u)); zp.write_bytes(next(iter(u.values())))
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,"rb") as f:
  for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
 return h.hexdigest()
if EXPECTED_ARTIFACT_ZIP_SHA256 and sha256_file(zp).lower()!=EXPECTED_ARTIFACT_ZIP_SHA256.strip().lower(): raise ValueError("Whole-ZIP SHA-256 mismatch")
def manifest_member_path(root,name):
 if "\\" in name: raise ValueError("Unsafe artifact path")
 q=PurePosixPath(name)
 if q.is_absolute() or ".." in q.parts: raise ValueError("Unsafe artifact path")
 p=(Path(root).resolve()/Path(*q.parts)).resolve()
 if p!=Path(root).resolve() and Path(root).resolve() not in p.parents: raise ValueError("Artifact path escape")
 return p
root=Path("/content/dimer-language-model-artifact"); shutil.rmtree(root,ignore_errors=True); root.mkdir()
total=0
with zipfile.ZipFile(zp) as z:
 for i in z.infolist():
  if stat.S_ISLNK((i.external_attr>>16)&0xffff): raise ValueError("Symlink not allowed")
  total+=i.file_size
  if total>512*1024**2: raise ValueError("Artifact exceeds 512 MiB")
  p=manifest_member_path(root,i.filename)
  if i.is_dir(): p.mkdir(parents=True,exist_ok=True); continue
  p.parent.mkdir(parents=True,exist_ok=True)
  with z.open(i) as src,open(p,"wb") as dst: shutil.copyfileobj(src,dst)
ms=list(root.rglob("artifact-manifest.json"))
if len(ms)!=1: raise ValueError("Expected one artifact-manifest.json")
artifact_root=ms[0].parent; m=json.loads(ms[0].read_text())
if m.get("format")!="peft_adapter" or m.get("formatVersion")!=1: raise ValueError("Unsupported artifact format")
listed=set(); size=0
for r in m.get("files",[]):
 p=manifest_member_path(artifact_root,r["path"])
 if not p.is_file() or p.stat().st_size!=r["bytes"] or sha256_file(p)!=r["sha256"]: raise ValueError(f"SHA-256 mismatch: {r['path']}")
 listed.add(r["path"]); size+=r["bytes"]
actual={p.relative_to(artifact_root).as_posix() for p in artifact_root.rglob("*") if p.is_file() and p.name!="artifact-manifest.json"}
if actual!=listed or size!=m.get("totalBytes"): raise ValueError("Manifest/file-set mismatch")
print("✓ Adapter manifest verified")


## 2. Resolve Immutable Base-Model Provenance

A PEFT adapter contains low-rank delta matrices rather than full model weights; it can only generate text when attached to the specific base model it was trained on.

The package's `provenance.json` records the exact base model identity, including its immutable 40-character commit revision:
- **Pinning**: Enforces that `baseModelRevision` is a valid 40-character SHA matching the expected commit, preventing silent drift from floating tags; and
- **Safety**: Verifies `trustRemoteCode=False` to ensure untrusted code is never executed.

If the base model is gated on Hugging Face (such as Llama 3.2), the notebook retrieves `HF_TOKEN` from Colab Secrets and performs an authenticated metadata preflight check before downloading weights.


In [ ]:
p=json.loads((artifact_root/"provenance.json").read_text())
if p.get("trustRemoteCode") is not False: raise ValueError("trustRemoteCode must be false")
base_model_id=p.get("baseModel"); base_revision=p.get("baseModelRevision")
if not re.fullmatch(r"[0-9a-f]{40}",base_revision or "") or p.get("baseModelRevisionExpected")!=base_revision: raise ValueError("Invalid baseModelRevision provenance")
BASE_MODEL_SOURCE = "Pinned Hugging Face" # @param ["Pinned Hugging Face","DIMER ZIP"]
DIMER_BASE_ZIP_PATH = "/content/dimer-base-model.zip" # @param {type:"string"}
requires_token=bool(p.get("requiresHfToken")); dimer_ok=bool(p.get("dimerZipAllowed",True)); model_key=p.get("modelKey")
if BASE_MODEL_SOURCE=="DIMER ZIP" and not dimer_ok: raise RuntimeError("This artifact does not permit/assume a DIMER-hosted base ZIP")
HF_TOKEN=None
if BASE_MODEL_SOURCE=="Pinned Hugging Face" and requires_token:
 from google.colab import userdata
 try: HF_TOKEN=userdata.get("HF_TOKEN")
 except Exception as exc: raise RuntimeError("Add HF_TOKEN in Colab Secrets and enable Notebook access") from exc
 info=HfApi(token=HF_TOKEN).model_info(base_model_id,revision=base_revision)
 if info.sha!=base_revision: raise RuntimeError("Pinned revision mismatch")
 print("✓ Hugging Face credential/revision preflight passed")


## 3. Base-Model Acquisition (Hugging Face or Offline Snapshot)

To attach the adapter, we need the base model weights. Two source options are supported:
- **`Pinned Hugging Face`**: Downloads the exact commit revision directly from the Hugging Face Hub; or
- **`DIMER ZIP`**: Loads a pre-packaged offline base model snapshot archive.

When loading an offline snapshot, the archive must contain a valid `dimer-base-manifest.json` file (`format: dimer_hf_snapshot`). The notebook verifies that the snapshot's model identity and commit revision match the adapter provenance, and that every file passes byte-count and SHA-256 integrity verification. The verified model is then loaded with `local_files_only=True` to guarantee offline reproducibility.


In [ ]:
def bmember(root,name):
 q=PurePosixPath(name)
 if "\\" in name or q.is_absolute() or ".." in q.parts: raise ValueError("Unsafe DIMER path")
 p=(Path(root).resolve()/Path(*q.parts)).resolve()
 if p!=Path(root).resolve() and Path(root).resolve() not in p.parents: raise ValueError("DIMER path escape")
 return p
BASE_MODEL_LOAD_REF=base_model_id
if BASE_MODEL_SOURCE=="DIMER ZIP":
 zpath=Path(DIMER_BASE_ZIP_PATH)
 r=Path("/content/dimer-inference-base"); shutil.rmtree(r,ignore_errors=True); r.mkdir(); total=0
 with zipfile.ZipFile(zpath) as z:
  for i in z.infolist():
   if stat.S_ISLNK((i.external_attr>>16)&0xffff): raise ValueError("Symlink not allowed")
   total+=i.file_size
   if total>20*1024**3: raise ValueError("DIMER ZIP too large")
   q=bmember(r,i.filename)
   if i.is_dir(): q.mkdir(parents=True,exist_ok=True); continue
   q.parent.mkdir(parents=True,exist_ok=True)
   with z.open(i) as src,open(q,"wb") as dst: shutil.copyfileobj(src,dst)
 ms=list(r.rglob("dimer-base-manifest.json"))
 if len(ms)!=1: raise ValueError("Expected one dimer-base-manifest.json")
 mr=ms[0].parent; bm=json.loads(ms[0].read_text())
 if bm.get("format")!="dimer_hf_snapshot" or (bm.get("modelKey"),bm.get("modelId"),bm.get("revision"))!=(model_key,base_model_id,base_revision): raise ValueError("DIMER base identity mismatch")
 for f in bm.get("files",[]):
  q=bmember(mr,f["path"])
  if not q.is_file() or q.stat().st_size!=f["bytes"] or sha256_file(q)!=f["sha256"]: raise ValueError(f"SHA-256 mismatch: {f['path']}")
 BASE_MODEL_LOAD_REF=mr; print("✓ DIMER base-model package verified")


## 4. Attach the Adapter and Generate

We load the base model in 4-bit precision (`nf4`) for memory efficiency, allowing inference to run smoothly even on entry-level GPUs. The tokenizer is loaded directly from the adapter bundle's `tokenizer/` directory, preserving the exact vocabulary and chat formatting used during fine-tuning.

Using `peft.PeftModel.from_pretrained()`, we attach the trained low-rank matrices to the base model in evaluation mode (`is_trainable=False`).

### Generation & Sampling Guidelines
When evaluating conversational models, output variability is controlled via decoding parameters:
- **Thinking Mode:** For models that generate `<think>...</think>` traces, use `temperature=0.6`, `top_p=0.95`, `top_k=20`. Avoid greedy search (`temperature=0.0`) in thinking mode to prevent repetitive reasoning loops.
- **Non-Thinking Mode:** For direct conversational answers, use `temperature=0.7`, `top_p=0.8`, `top_k=20`.
- **Deterministic Smoke Tests:** Set `do_sample=False` (greedy decoding) to ensure deterministic, reproducible output comparisons.

Testing generation with sample prompts demonstrates that:
1. The exported adapter artifact is completely self-contained;
2. The tokenizer chat template applies correctly to format multi-turn conversations; and
3. The adapted model generates responses reflecting its fine-tuned behavior.


In [ ]:
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
tok=AutoTokenizer.from_pretrained(artifact_root/"tokenizer",local_files_only=True,trust_remote_code=False)
kw={"trust_remote_code":False,"dtype":dtype,"quantization_config":q,"device_map":{"":0}}
if BASE_MODEL_SOURCE=="Pinned Hugging Face": kw.update({"revision":base_revision,**({"token":HF_TOKEN} if HF_TOKEN else {})})
else: kw["local_files_only"]=True
base=AutoModelForCausalLM.from_pretrained(BASE_MODEL_LOAD_REF,**kw)
model=PeftModel.from_pretrained(base,artifact_root,is_trainable=False)
def reply(prompt):
 s=tok.apply_chat_template([{"role":"user","content":prompt}],tokenize=False,add_generation_prompt=True); x=tok(s,return_tensors="pt",add_special_tokens=False).to("cuda")
 with torch.no_grad(): y=model.generate(**x,max_new_tokens=96,do_sample=False,pad_token_id=tok.pad_token_id)
 return tok.decode(y[0,x["input_ids"].shape[1]:],skip_special_tokens=True).strip()
PROMPTS=["Ipaliwanag sa simpleng Filipino kung ano ang machine learning.","Sumulat ng maikling payo para sa isang estudyanteng nagsisimula sa AI."]
display(pd.DataFrame({"prompt":PROMPTS,"response":[reply(x) for x in PROMPTS]}))
print("✓ Artifact verified, base resolved, adapter attached, generation complete")


## Result and Next Steps

The fine-tuned adapter package has successfully cleared all integrity and provenance checks, attached to its base model, and generated responses—all without requiring the original training dataset.

---

### Production Serving Architectures

For production deployment, you can choose from three industry-standard serving patterns:

1. **High-Throughput Multi-LoRA Serving (vLLM / SGLang / TGI):**
   High-throughput inference engines support dynamic multi-LoRA routing, enabling hundreds of specialized fine-tuned adapters to run concurrently on top of a single shared base model instance with minimal GPU memory overhead.
2. **Local Python Integration (PEFT & Transformers):**
   Use Hugging Face `peft` and `transformers` as demonstrated in this notebook for embedded tools, batch scoring, desktop applications, or Model Context Protocol (MCP) agent tools.
3. **Zero-Overhead Weight Merging:**
   Call `model.merge_and_unload()` to permanently fold the low-rank delta weights directly into the base model parameters. This exports a unified, standard Hugging Face checkpoint loadable by any engine (llama.cpp, Ollama, ONNX, vLLM) with zero adapter latency overhead.
